
# Reproducible baselines

In [ ]:
#Imports
from google.colab import drive
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
from pathlib import Path


# Leakage-safe model inputs

Verifying dataset

In [ ]:
df = pd.read_csv("/content/allstate_claims_data.csv")
manifest = pd.read_csv("/content/gate1_partition_manifest.csv")

print(df.shape)
print(manifest.shape)

(188318, 132)
(188318, 2)


In [ ]:

print("Dataset shape:", df.shape)
print("Manifest shape:", manifest.shape)

print("\nPartition counts:")
print(manifest["partition"].value_counts())

assert {"id", "loss"}.issubset(df.columns)
assert {"id", "partition"}.issubset(manifest.columns)
assert manifest["id"].is_unique
assert len(manifest) == len(df)
assert set(manifest["id"]) == set(df["id"])
assert set(manifest["partition"]) == {"train", "validation", "test"}

print("\nPASS: Dataset and manifest verified.")


Dataset shape: (188318, 132)
Manifest shape: (188318, 2)

Partition counts:
partition
train         131822
validation     28248
test           28248
Name: count, dtype: int64

PASS: Dataset and manifest verified.


In [ ]:

import numpy as np
from sklearn.metrics import mean_absolute_error

# Reconstruct the frozen partitions from the manifest
train_ids = manifest.loc[
    manifest["partition"] == "train", "id"
]
val_ids = manifest.loc[
    manifest["partition"] == "validation", "id"
]

train_df = df[df["id"].isin(train_ids)].copy()
val_df = df[df["id"].isin(val_ids)].copy()

# Predict the training partition's mean loss for every validation claim
training_mean = train_df["loss"].mean()
mean_predictions = np.full(len(val_df), training_mean)

mean_mae = mean_absolute_error(
    val_df["loss"],
    mean_predictions
)

print("Training mean:", training_mean)
print("Validation claims:", len(val_df))
print("Mean-constant baseline MAE:", mean_mae)

Training mean: 3033.6880441807893
Validation claims: 28248
Mean-constant baseline MAE: 1978.3970470120255


In [ ]:

# Median-constant comparator

training_median = train_df["loss"].median()

median_predictions = np.full(
    len(val_df),
    training_median
)

median_mae = mean_absolute_error(
    val_df["loss"],
    median_predictions
)

print("Training median:", training_median)
print("Validation claims:", len(val_df))
print("Median baseline MAE:", median_mae)


Training median: 2113.755
Validation claims: 28248
Median baseline MAE: 1827.3136338855847


Validate prediction-to-ID alignment

In [ ]:

# Gate 2: Baseline report, validation checks, and run receipt

from pathlib import Path
import hashlib
import json
import subprocess
import sys
from datetime import datetime, timezone

# 1. Align predictions explicitly with validation claim IDs
mean_results = pd.DataFrame({
    "id": val_df["id"].to_numpy(),
    "prediction": np.asarray(mean_predictions, dtype=float)
})

median_results = pd.DataFrame({
    "id": val_df["id"].to_numpy(),
    "prediction": np.asarray(median_predictions, dtype=float)
})

assert mean_results["id"].equals(median_results["id"])
assert mean_results["id"].equals(val_df["id"].reset_index(drop=True))
assert mean_results["id"].is_unique
assert len(mean_results) == len(val_df)
assert len(median_results) == len(val_df)
assert np.isfinite(mean_results["prediction"]).all()
assert np.isfinite(median_results["prediction"]).all()

# 2. Independently calculate MAE using NumPy
y_val = val_df["loss"].to_numpy(dtype=float)

mean_mae_numpy = float(
    np.mean(np.abs(y_val - mean_results["prediction"].to_numpy()))
)
median_mae_numpy = float(
    np.mean(np.abs(y_val - median_results["prediction"].to_numpy()))
)

assert np.isclose(mean_mae, mean_mae_numpy, atol=1e-10)
assert np.isclose(median_mae, median_mae_numpy, atol=1e-10)

# 3. Identify and fingerprint the frozen partition manifest
manifest_path = Path("/content/gate1_partition_manifest.csv")

if manifest_path.is_file():
    manifest_sha256 = hashlib.sha256(
        manifest_path.read_bytes()
    ).hexdigest()
    manifest_id = f"sha256:{manifest_sha256}"
else:
    # Fingerprint the in-memory manifest if the CSV is elsewhere.
    manifest_bytes = manifest.sort_values("id").to_csv(
        index=False
    ).encode("utf-8")
    manifest_id = "in-memory-sha256:" + hashlib.sha256(
        manifest_bytes
    ).hexdigest()

# 4. Fingerprint the source dataset
source_candidates = [
    Path("/content/allstate_claims_data.csv"),
    Path("/content/drive/MyDrive/allstate_claims_data.csv")
]
source_file = next(
    (p for p in source_candidates if p.is_file()), None
)

source_sha256 = (
    hashlib.sha256(source_file.read_bytes()).hexdigest()
    if source_file else None
)

# 5. Record code version where available
try:
    code_version = subprocess.check_output(
        ["git", "rev-parse", "HEAD"],
        stderr=subprocess.DEVNULL,
        text=True
    ).strip()
except Exception:
    code_version = "Git commit unavailable in this Colab session"

# 6. Build report and run receipt
report = {
    "gate": "Gate 2 baseline verification",
    "run_timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "partition_manifest_id": manifest_id,
    "source_file": str(source_file) if source_file else
        "Source path not resolved; see source SHA-256 field",
    "source_sha256": source_sha256,
    "code_version": code_version,
    "python_version": sys.version,
    "pandas_version": pd.__version__,
    "numpy_version": np.__version__,
    "sklearn_version": __import__("sklearn").__version__,
    "split_seed": 1,
    "validation_claim_count": len(val_df),
    "target": "loss",
    "target_transformation": "None; predictions in original loss units",
    "metric": "Mean Absolute Error (MAE)",
    "prediction_checks": {
        "one_prediction_per_validation_claim":
            len(mean_results) == len(val_df)
            and len(median_results) == len(val_df),
        "prediction_ids_aligned":
            mean_results["id"].equals(val_df["id"].reset_index(drop=True))
            and median_results["id"].equals(val_df["id"].reset_index(drop=True)),
        "validation_ids_unique": bool(val_df["id"].is_unique),
        "mean_predictions_finite":
            bool(np.isfinite(mean_results["prediction"]).all()),
        "median_predictions_finite":
            bool(np.isfinite(median_results["prediction"]).all()),
        "mean_mae_independently_verified":
            bool(np.isclose(mean_mae, mean_mae_numpy, atol=1e-10)),
        "median_mae_independently_verified":
            bool(np.isclose(median_mae, median_mae_numpy, atol=1e-10))
    },
    "baselines": {
        "mean_constant": {
            "fitted_constant": float(training_mean),
            "validation_mae": float(mean_mae),
            "independent_numpy_mae": mean_mae_numpy
        },
        "median_constant_comparator": {
            "fitted_constant": float(training_median),
            "validation_mae": float(median_mae),
            "independent_numpy_mae": median_mae_numpy
        }
    }
}

# Save reproducible artifacts
mean_results.to_csv(
    "/content/gate2_mean_baseline_predictions.csv", index=False
)
median_results.to_csv(
    "/content/gate2_median_baseline_predictions.csv", index=False
)

with open("/content/gate2_baseline_run_receipt.json", "w") as f:
    json.dump(report, f, indent=2)

# 7. Print final report
print("GATE 2 BASELINE REPORT")
print("=" * 60)
print("Partition manifest ID:", manifest_id)
print("Validation claims:", len(val_df))
print("Target / units: loss / original loss units")
print("Source SHA-256:", source_sha256)
print("Code version:", code_version)

print("\nMean-constant baseline")
print("Fitted constant:", training_mean)
print("Validation MAE:", mean_mae)
print("Independent NumPy MAE:", mean_mae_numpy)

print("\nMedian-constant comparator")
print("Fitted constant:", training_median)
print("Validation MAE:", median_mae)
print("Independent NumPy MAE:", median_mae_numpy)

print("\nValidation checks:")
for check, passed in report["prediction_checks"].items():
    print(f"{check}: {'PASS' if passed else 'FAIL'}")

assert all(report["prediction_checks"].values())

print("\nPASS: Both baselines independently verified.")
print("Run receipt: /content/gate2_baseline_run_receipt.json")
print("Prediction files saved under /content/")


GATE 2 BASELINE REPORT
Partition manifest ID: sha256:0e97b61a8ad550387f2c7bcbe2b4130064e790337e88d1a5251399ce91830418
Validation claims: 28248
Target / units: loss / original loss units
Source SHA-256: 74037cb248a1064e4d578692a4f4e5d8492ed1b2033daf643496e1b68b14ae03
Code version: Git commit unavailable in this Colab session

Mean-constant baseline
Fitted constant: 3033.6880441807893
Validation MAE: 1978.3970470120255
Independent NumPy MAE: 1978.3970470120255

Median-constant comparator
Fitted constant: 2113.755
Validation MAE: 1827.3136338855847
Independent NumPy MAE: 1827.3136338855847

Validation checks:
one_prediction_per_validation_claim: PASS
prediction_ids_aligned: PASS
validation_ids_unique: PASS
mean_predictions_finite: PASS
median_predictions_finite: PASS
mean_mae_independently_verified: PASS
median_mae_independently_verified: PASS

PASS: Both baselines independently verified.
Run receipt: /content/gate2_baseline_run_receipt.json
Prediction files saved under /content/


In [ ]:
# List every column and its data type
column_audit = pd.DataFrame({
    "column": df.columns,
    "dtype": df.dtypes.astype(str).values,
    "in_training_data": [c in train_df.columns for c in df.columns],
    "in_validation_data": [c in val_df.columns for c in df.columns]
})

print("Total columns:", len(column_audit))
print("\nTarget and potentially target-derived columns:")
print(
    column_audit[
        column_audit["column"].str.contains(
            r"(^id$|^loss$|log1p_loss|target|prediction|mean|median|severity)",
            case=False,
            regex=True
        )
    ].to_string(index=False)
)

print("\nAll columns:")
print(column_audit.to_string(index=False))

Total columns: 132

Target and potentially target-derived columns:
column   dtype  in_training_data  in_validation_data
    id   int64              True                True
  loss float64              True                True

All columns:
column   dtype  in_training_data  in_validation_data
    id   int64              True                True
  cat1  object              True                True
  cat2  object              True                True
  cat3  object              True                True
  cat4  object              True                True
  cat5  object              True                True
  cat6  object              True                True
  cat7  object              True                True
  cat8  object              True                True
  cat9  object              True                True
 cat10  object              True                True
 cat11  object              True                True
 cat12  object              True                True
 cat13  object    

/tmp/ipykernel_817/547265813.py:13: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  column_audit["column"].str.contains(


In [ ]:
# Define candidate inputs without ID or target leakage
excluded_columns = ["id", "loss"]

feature_columns = [
    column for column in train_df.columns
    if column not in excluded_columns
]

X_train_raw = train_df[feature_columns].copy()
X_val_raw = val_df[feature_columns].copy()

y_train = train_df["loss"].copy()
y_val = val_df["loss"].copy()

assert "id" not in X_train_raw.columns
assert "loss" not in X_train_raw.columns
assert "id" not in X_val_raw.columns
assert "loss" not in X_val_raw.columns
assert list(X_train_raw.columns) == list(X_val_raw.columns)

print("Candidate input features:", len(feature_columns))
print("Training feature shape:", X_train_raw.shape)
print("Validation feature shape:", X_val_raw.shape)
print("Target excluded from inputs:", "loss" not in X_train_raw.columns)
print("ID excluded from inputs:", "id" not in X_train_raw.columns)

Candidate input features: 130
Training feature shape: (131822, 130)
Validation feature shape: (28248, 130)
Target excluded from inputs: True
ID excluded from inputs: True


In [ ]:
# Validate prediction alignment, counts, finiteness, and MAE independently

# 1. Prediction-to-ID alignment
assert np.array_equal(
    mean_predictions, np.full(len(val_df), training_mean)
)
assert np.array_equal(
    median_predictions, np.full(len(val_df), training_median)
)

# 2. One prediction per validation claim
assert len(mean_predictions) == len(val_df)
assert len(median_predictions) == len(val_df)
assert val_df["id"].is_unique

# 3. Finite predictions
assert np.isfinite(mean_predictions).all()
assert np.isfinite(median_predictions).all()

# 4. Independently calculate MAE using NumPy
mean_mae_independent = np.mean(
    np.abs(val_df["loss"].to_numpy() - mean_predictions)
)
median_mae_independent = np.mean(
    np.abs(val_df["loss"].to_numpy() - median_predictions)
)

assert np.isclose(mean_mae, mean_mae_independent, rtol=1e-12)
assert np.isclose(median_mae, median_mae_independent, rtol=1e-12)

# Report results
print("VALIDATION RESULTS")
print("=" * 45)

print("Mean baseline:")
print("  Prediction-to-ID alignment: PASS")
print("  One prediction per claim:", len(mean_predictions) == len(val_df))
print("  All predictions finite:", np.isfinite(mean_predictions).all())
print(f"  sklearn MAE:       {mean_mae:.12f}")
print(f"  Independent MAE:   {mean_mae_independent:.12f}")
print("  MAE accuracy: PASS")

print("\nMedian baseline:")
print("  Prediction-to-ID alignment: PASS")
print("  One prediction per claim:", len(median_predictions) == len(val_df))
print("  All predictions finite:", np.isfinite(median_predictions).all())
print(f"  sklearn MAE:       {median_mae:.12f}")
print(f"  Independent MAE:   {median_mae_independent:.12f}")
print("  MAE accuracy: PASS")

print("\nAll validation checks passed.")

VALIDATION RESULTS
Mean baseline:
  Prediction-to-ID alignment: PASS
  One prediction per claim: True
  All predictions finite: True
  sklearn MAE:       1978.397047012025
  Independent MAE:   1978.397047012025
  MAE accuracy: PASS

Median baseline:
  Prediction-to-ID alignment: PASS
  One prediction per claim: True
  All predictions finite: True
  sklearn MAE:       1827.313633885585
  Independent MAE:   1827.313633885585
  MAE accuracy: PASS

All validation checks passed.



# Leakage-safe model inputs

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Identify feature types using training data only
categorical_features = X_train_raw.select_dtypes(
    include=["object", "category"]
).columns.tolist()

numeric_features = X_train_raw.select_dtypes(
    include=["number"]
).columns.tolist()

# Numeric preprocessing: training-set median imputation
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

# Categorical preprocessing: training-set mode imputation, then one-hot encoding
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Combine preprocessing steps
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

# Fit ONLY on training features
X_train_processed = preprocessor.fit_transform(X_train_raw)

# Apply the fitted transformations unchanged to validation features
X_val_processed = preprocessor.transform(X_val_raw)

print("Categorical input columns:", len(categorical_features))
print("Numeric input columns:", len(numeric_features))
print("Processed training shape:", X_train_processed.shape)
print("Processed validation shape:", X_val_processed.shape)
print("Preprocessing fitted on training data only: YES")
print("Validation transformed without refitting: YES")

Categorical input columns: 116
Numeric input columns: 14
Processed training shape: (131822, 1119)
Processed validation shape: (28248, 1119)
Preprocessing fitted on training data only: YES
Validation transformed without refitting: YES


In [ ]:
# Recalculate categorical feature-to-target relationships using training data only

categorical_features = train_df.select_dtypes(
    include=["object", "category"]
).columns.tolist()

target_relationships = []

for feature in categorical_features:
    stats = (
        train_df.groupby(feature, dropna=False)["loss"]
        .agg(["count", "mean", "median"])
        .reset_index()
    )

    stats["feature"] = feature
    stats = stats.rename(columns={feature: "category"})

    target_relationships.append(stats)

target_relationships_df = pd.concat(
    target_relationships,
    ignore_index=True
)

print("Training claims analyzed:", len(train_df))
print("Categorical features analyzed:", len(categorical_features))
print("Category-level rows analyzed:", len(target_relationships_df))
print("\nExample results:")
print(target_relationships_df.head(20).to_string(index=False))

Training claims analyzed: 131822
Categorical features analyzed: 116
Category-level rows analyzed: 1105

Example results:
category  count        mean   median feature
       A  99178 3401.731015 2452.295    cat1
       B  32644 1915.511174 1449.745    cat1
       A  74759 2450.617263 1740.660    cat2
       B  57063 3797.576878 2829.350    cat2
       A 124626 2901.254073 2034.245    cat3
       B   7196 5327.283948 4531.595    cat3
       A  89899 2825.756966 1977.310    cat4
       B  41923 3479.572069 2452.270    cat4
       A  86566 2814.664415 1949.090    cat5
       B  45256 3452.638006 2447.710    cat5
       A  92257 3260.776966 2335.800    cat6
       B  39565 2504.165925 1636.170    cat6
       A 128607 2902.264415 2071.220    cat7
       B   3215 8290.919359 7673.470    cat7
       A 124104 2971.709159 2068.465    cat8
       B   7718 4030.296962 3037.760    cat8
       A  79280 2510.245523 1780.100    cat9
       B  52542 3823.504250 2852.500    cat9
       A 112110 2692.199

In [ ]:
# Recheck September's target-related findings using October training data only

# Categorical fields highlighted in September
review_cats = [
    "cat57", "cat89", "cat7",
    "cat116", "cat110", "cat109"
]

for feature in review_cats:
    summary = (
        train_df.groupby(feature, dropna=False)["loss"]
        .agg(
            support="count",
            mean_loss="mean",
            median_loss="median",
            p95_loss=lambda x: x.quantile(0.95)
        )
        .sort_values("support", ascending=False)
    )

    print(f"\n{feature}: October training data only")
    print(summary.to_string())

# Recalculate continuous-feature correlations on training data only
continuous_features = [f"cont{i}" for i in range(1, 15)]

train_correlations = train_df[continuous_features].corrwith(
    train_df["loss"],
    method="pearson"
)

print("\nContinuous-feature correlations with loss")
print(
    train_correlations
    .reindex(train_correlations.abs().sort_values(ascending=False).index)
    .to_string()
)


cat57: October training data only
       support     mean_loss  median_loss   p95_loss
cat57                                               
A       129669   2913.563620      2079.48   7907.654
B         2153  10268.436772      9497.31  19469.652

cat89: October training data only
       support    mean_loss  median_loss    p95_loss
cat89                                               
A       128607  2902.264415      2071.22   7876.3370
B         3036  8292.027487      7731.97  17817.7675
C          151  8413.915762      7099.26  17937.5800
D           22  8049.927727      6303.90  15892.7115
E            3  3271.863333      2888.69   4116.1460
I            2  9814.040000      9814.04  16673.2280
G            1  3666.930000      3666.93   3666.9300

cat7: October training data only
      support    mean_loss  median_loss   p95_loss
cat7                                              
A      128607  2902.264415      2071.22   7876.337
B        3215  8290.919359      7673.47  17817.951

ca

In [ ]:


# Define rare categories using training data only.
# A category is rare if it appears fewer than 10 times in training.
rare_threshold = 10

categorical_features = X_train_raw.select_dtypes(
    include=["object", "category"]
).columns.tolist()

rare_categories = {}
unseen_categories = {}
rare_counts_by_feature = {}
unseen_counts_by_feature = {}

for col in categorical_features:
    train_counts = X_train_raw[col].value_counts(dropna=False)

    # Categories seen in training fewer than 10 times
    rare_categories[col] = set(
        train_counts[train_counts < rare_threshold].index
    )

    # Categories in validation that never appeared in training
    train_values = set(X_train_raw[col].dropna().unique())
    val_values = set(X_val_raw[col].dropna().unique())
    unseen_categories[col] = val_values - train_values

    # Count validation claims containing a rare or unseen value
    rare_mask = X_val_raw[col].isin(rare_categories[col])
    unseen_mask = X_val_raw[col].isin(unseen_categories[col])

    rare_counts_by_feature[col] = int(rare_mask.sum())
    unseen_counts_by_feature[col] = int(unseen_mask.sum())

# Summarize affected validation claims by feature
category_report = pd.DataFrame({
    "rare_training_levels": {
        col: len(rare_categories[col]) for col in categorical_features
    },
    "validation_claims_with_rare_value": rare_counts_by_feature,
    "unseen_validation_levels": {
        col: len(unseen_categories[col]) for col in categorical_features
    },
    "validation_claims_with_unseen_value": unseen_counts_by_feature
})

# Count validation claims affected by at least one rare or unseen value
any_rare = pd.Series(False, index=X_val_raw.index)
any_unseen = pd.Series(False, index=X_val_raw.index)

for col in categorical_features:
    any_rare |= X_val_raw[col].isin(rare_categories[col])
    any_unseen |= X_val_raw[col].isin(unseen_categories[col])

any_affected = any_rare | any_unseen

print(f"Rare-category threshold: fewer than {rare_threshold} training occurrences")
print(f"Validation claims: {len(X_val_raw):,}")
print(f"Claims with at least one rare category: {any_rare.sum():,}")
print(f"Claims with at least one unseen category: {any_unseen.sum():,}")
print(f"Claims with either issue: {any_affected.sum():,}")
print(f"Claims with neither issue: {(~any_affected).sum():,}")

print("\nFeatures with the most affected validation claims:")
display(
    category_report.sort_values(
        "validation_claims_with_unseen_value",
        ascending=False
    ).head(15)
)

Rare-category threshold: fewer than 10 training occurrences
Validation claims: 28,248
Claims with at least one rare category: 139
Claims with at least one unseen category: 23
Claims with either issue: 158
Claims with neither issue: 28,090

Features with the most affected validation claims:


,rare_training_levels,validation_claims_with_rare_value,unseen_validation_levels,validation_claims_with_unseen_value
cat116,104,63,12,16
cat104,0,0,1,3
cat109,26,18,2,3
cat105,4,3,2,2
cat92,1,2,1,1
cat114,5,4,1,1
cat113,11,11,1,1
cat110,39,33,1,1
cat9,0,0,0,0
cat10,0,0,0,0


Using a threshold of fewer than 10 occurrences in the training data, 139 validation claims contained at least one rare category, while 23 contained at least one unseen category. Overall, 158 unique validation claims (0.56% of the validation set) were affected by either issue, while 28,090 were unaffected. Cat116 contributed the most rare and unseen category cases. The existing OneHotEncoder(handle_unknown="ignore") handles unseen categories, and all rules were established without using validation loss.

In [ ]:
# Audit model inputs for leakage and disputed fields

forbidden_exact = {"id", "loss", "log1p_loss"}

# Catch common names for target-derived summaries
target_summary_terms = (
    "mean_loss", "median_loss", "p95_loss",
    "target_mean", "target_encoding", "loss_mean",
    "loss_median", "target_summary"
)

train_columns = set(X_train_raw.columns)
val_columns = set(X_val_raw.columns)

print("1. Forbidden fields in training inputs:")
print(sorted(col for col in train_columns if col.lower() in forbidden_exact))

print("\n2. Forbidden fields in validation inputs:")
print(sorted(col for col in val_columns if col.lower() in forbidden_exact))

print("\n3. Potential target-summary fields in training inputs:")
print(sorted(
    col for col in X_train_raw.columns
    if any(term in col.lower() for term in target_summary_terms)
))

print("\n4. Potential target-summary fields in validation inputs:")
print(sorted(
    col for col in X_val_raw.columns
    if any(term in col.lower() for term in target_summary_terms)
))

print("\n5. Do training and validation feature columns match?")
print(train_columns == val_columns)

if train_columns != val_columns:
    print("Only in training:", sorted(train_columns - val_columns))
    print("Only in validation:", sorted(val_columns - train_columns))

print("\n6. Feature counts:")
print("Training features:", len(X_train_raw.columns))
print("Validation features:", len(X_val_raw.columns))
print("Training rows:", len(X_train_raw))
print("Validation rows:", len(X_val_raw))

1. Forbidden fields in training inputs:
[]

2. Forbidden fields in validation inputs:
[]

3. Potential target-summary fields in training inputs:
[]

4. Potential target-summary fields in validation inputs:
[]

5. Do training and validation feature columns match?
True

6. Feature counts:
Training features: 130
Validation features: 130
Training rows: 131822
Validation rows: 28248


Verify how X_train_raw and X_val_raw were constructed and whether they contain only original predictor columns

In [ ]:
# Check whether model inputs come from the original dataset
original_predictors = [
    col for col in df.columns
    if col not in ["id", "loss", "log1p_loss"]
]

print("Original predictor count:", len(original_predictors))
print("Training input count:", len(X_train_raw.columns))
print("Validation input count:", len(X_val_raw.columns))

print("\nTraining inputs match original predictors:",
      set(X_train_raw.columns) == set(original_predictors))

print("Validation inputs match original predictors:",
      set(X_val_raw.columns) == set(original_predictors))

print("\nColumns in training inputs that are not original predictors:",
      sorted(set(X_train_raw.columns) - set(original_predictors)))

print("\nOriginal predictors missing from training inputs:",
      sorted(set(original_predictors) - set(X_train_raw.columns)))

Original predictor count: 130
Training input count: 130
Validation input count: 130

Training inputs match original predictors: True
Validation inputs match original predictors: True

Columns in training inputs that are not original predictors: []

Original predictors missing from training inputs: []


The audit passes :)

**Document any target transformation and its inverse-prediction rule. Calculate MAE in original loss units and record how negative or non-finite predictions are handled**

In [ ]:
import numpy as np
from sklearn.metrics import mean_absolute_error

# 1. Document the target transformation
# Current y_train and y_val are expected to contain original loss values.
target_transformation = "None: model target is original loss"
inverse_prediction_rule = "Identity: predictions are already in loss units"

print("Target transformation:", target_transformation)
print("Inverse-prediction rule:", inverse_prediction_rule)
print("Training target range:", float(y_train.min()), "to", float(y_train.max()))
print("Validation target range:", float(y_val.min()), "to", float(y_val.max()))

# 2. Define prediction handling for all models
# Replace non-finite predictions with the training-set median.
# Clip negative predictions to zero.
training_loss_median = float(np.median(y_train))

def clean_predictions(predictions):
    predictions = np.asarray(predictions, dtype=float).copy()
    predictions[~np.isfinite(predictions)] = training_loss_median
    predictions = np.maximum(predictions, 0)
    return predictions

# 3. Calculate baseline MAE in original loss units
mean_baseline = np.full(len(y_val), np.mean(y_train))
median_baseline = np.full(len(y_val), training_loss_median)

print("\nBaseline MAE in original loss units:")
print("Mean baseline:", mean_absolute_error(
    y_val, clean_predictions(mean_baseline)
))
print("Median baseline:", mean_absolute_error(
    y_val, clean_predictions(median_baseline)
))

print("\nPrediction-handling rules:")
print("Non-finite predictions -> training-set median loss")
print("Negative predictions -> clipped to zero")
print("MAE is calculated against original validation loss")

Target transformation: None: model target is original loss
Inverse-prediction rule: Identity: predictions are already in loss units
Training target range: 0.67 to 121012.25
Validation target range: 54.34 to 51383.04

Baseline MAE in original loss units:
Mean baseline: 1978.3970470120255
Median baseline: 1827.3136338855847

Prediction-handling rules:
Non-finite predictions -> training-set median loss
Negative predictions -> clipped to zero
MAE is calculated against original validation loss


- Target transformation: None. Models predict the original loss values.
- Inverse-prediction rule: Identity. No inverse transformation is needed.
- Negative predictions: Clip to zero.
- Non-finite predictions: Replace with the training-set median loss, approximately 2,113.76.
- Evaluation metric: Mean Absolute Error (MAE), calculated in original loss units.
Baseline	Validation MAE
Training mean	1,978.40
Training median	1,827.31

The median baseline is better by approximately 151.08 loss units.


**Independently rerun baseline and pipeline checks using recorded source, code, split, and configuration. Resolve discrepancies and obtain Gate 2 approval before comparing feature-based models**

In [ ]:

import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error

# 1. Reload the recorded source files independently
source_check = pd.read_csv("/content/allstate_claims_data.csv")
manifest_check = pd.read_csv("/content/gate1_partition_manifest.csv")

print("=== SOURCE CHECK ===")
print("Dataset shape:", source_check.shape)
print("Manifest shape:", manifest_check.shape)
print("Dataset columns match current data:",
      list(source_check.columns) == list(df.columns))
print("Manifest columns:", manifest_check.columns.tolist())
print("Missing values in loss:", source_check["loss"].isna().sum())
print("Duplicate IDs in source:", source_check["id"].duplicated().sum())

# 2. Inspect the recorded split configuration
print("\n=== SPLIT MANIFEST CHECK ===")
print(manifest_check.head())
print("\nManifest column data types:")
print(manifest_check.dtypes)

# 3. Independently recalculate the baseline MAE
y_train_check = np.asarray(y_train, dtype=float)
y_val_check = np.asarray(y_val, dtype=float)

train_median_check = np.median(y_train_check)
train_mean_check = np.mean(y_train_check)

mean_predictions_check = np.full(
    len(y_val_check), train_mean_check
)
median_predictions_check = np.full(
    len(y_val_check), train_median_check
)

mean_mae_check = mean_absolute_error(
    y_val_check, mean_predictions_check
)
median_mae_check = mean_absolute_error(
    y_val_check, median_predictions_check
)

print("\n=== INDEPENDENT BASELINE CHECK ===")
print("Training rows:", len(y_train_check))
print("Validation rows:", len(y_val_check))
print("Training mean:", train_mean_check)
print("Training median:", train_median_check)
print("Mean baseline MAE:", mean_mae_check)
print("Median baseline MAE:", median_mae_check)

print("\nExpected previous results:")
print("Mean baseline MAE: 1978.397047")
print("Median baseline MAE: 1827.313634")


=== SOURCE CHECK ===
Dataset shape: (188318, 132)
Manifest shape: (188318, 2)
Dataset columns match current data: True
Manifest columns: ['id', 'partition']
Missing values in loss: 0
Duplicate IDs in source: 0

=== SPLIT MANIFEST CHECK ===
       id partition
0  238427     train
1  110608     train
2  587400     train
3  325126     train
4  127688     train

Manifest column data types:
id            int64
partition    object
dtype: object

=== INDEPENDENT BASELINE CHECK ===
Training rows: 131822
Validation rows: 28248
Training mean: 3033.6880441807893
Training median: 2113.755
Mean baseline MAE: 1978.3970470120255
Median baseline MAE: 1827.3136338855847

Expected previous results:
Mean baseline MAE: 1978.397047
Median baseline MAE: 1827.313634


In [ ]:

from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.utils.validation import check_is_fitted
from scipy import sparse

# 1. Verify source IDs and manifest assignments
source_ids = set(source_check["id"])
manifest_ids = set(manifest_check["id"])

print("=== SPLIT MEMBERSHIP CHECK ===")
print("Source IDs equal manifest IDs:", source_ids == manifest_ids)
print("Duplicate manifest IDs:", manifest_check["id"].duplicated().sum())
print("\nPartition counts:")
print(manifest_check["partition"].value_counts().sort_index())

train_ids = set(train_df["id"])
val_ids = set(val_df["id"])

train_manifest_ids = set(
    manifest_check.loc[manifest_check["partition"] == "train", "id"]
)
val_manifest_ids = set(
    manifest_check.loc[manifest_check["partition"] == "validation", "id"]
)

print("Training IDs match manifest:",
      train_ids == train_manifest_ids)
print("Validation IDs match manifest:",
      val_ids == val_manifest_ids)
print("Training and validation IDs overlap:",
      bool(train_ids & val_ids))

# 2. Check the fitted preprocessing pipeline
print("\n=== PREPROCESSOR CHECK ===")
check_is_fitted(preprocessor)
print("Preprocessor is fitted: True")
print("Input feature count:", len(X_train_raw.columns))
print("Training/validation feature columns match:",
      list(X_train_raw.columns) == list(X_val_raw.columns))

# Inspect configured transformers
params = preprocessor.get_params(deep=True)
encoders = [
    (name, obj) for name, obj in params.items()
    if isinstance(obj, OneHotEncoder)
]
imputers = [
    (name, obj) for name, obj in params.items()
    if isinstance(obj, SimpleImputer)
]

print("OneHotEncoder configurations:",
      [(name, obj.handle_unknown) for name, obj in encoders])
print("Imputer configurations:",
      [(name, obj.strategy) for name, obj in imputers])

# Rerun transformations
X_train_check = preprocessor.transform(X_train_raw)
X_val_check = preprocessor.transform(X_val_raw)

def all_finite(matrix):
    values = matrix.data if sparse.issparse(matrix) else np.asarray(matrix)
    return bool(np.isfinite(values).all())

print("\n=== TRANSFORM OUTPUT CHECK ===")
print("Training transformed shape:", X_train_check.shape)
print("Validation transformed shape:", X_val_check.shape)
print("Transformed feature counts match:",
      X_train_check.shape[1] == X_val_check.shape[1])
print("Training output is finite:", all_finite(X_train_check))
print("Validation output is finite:", all_finite(X_val_check))


=== SPLIT MEMBERSHIP CHECK ===
Source IDs equal manifest IDs: True
Duplicate manifest IDs: 0

Partition counts:
partition
test           28248
train         131822
validation     28248
Name: count, dtype: int64
Training IDs match manifest: True
Validation IDs match manifest: True
Training and validation IDs overlap: False

=== PREPROCESSOR CHECK ===
Preprocessor is fitted: True
Input feature count: 130
Training/validation feature columns match: True
OneHotEncoder configurations: [('categorical__encoder', 'ignore')]
Imputer configurations: [('numeric__imputer', 'median'), ('categorical__imputer', 'most_frequent')]

=== TRANSFORM OUTPUT CHECK ===
Training transformed shape: (131822, 1119)
Validation transformed shape: (28248, 1119)
Transformed feature counts match: True
Training output is finite: True
Validation output is finite: True


In [ ]:
from sklearn.base import clone
from scipy import sparse
import numpy as np

# Refit a fresh preprocessor using training data only
preprocessor_check = clone(preprocessor)
preprocessor_check.fit(X_train_raw)

# Transform training and validation data
X_train_verified = preprocessor_check.transform(X_train_raw)
X_val_verified = preprocessor_check.transform(X_val_raw)

# Compare fresh outputs against existing outputs
def same_values(a, b):
    if a.shape != b.shape:
        return False
    if sparse.issparse(a):
        a = a.toarray()
    if sparse.issparse(b):
        b = b.toarray()
    return bool(np.allclose(a, b))

print("Training transformation matches existing:",
      same_values(X_train_verified, X_train_processed))

print("Validation transformation matches existing:",
      same_values(X_val_verified, X_val_processed))

Training transformation matches existing: True
Validation transformation matches existing: True
